# Frontload-CL SFT smoke test (1×A100)

This notebook tests the SFT path that failed on the platform, without submitting a run:

1. reads a **gzip conversation shard whose cached filename has no `.gz` suffix**;
2. tokenizes it with the pinned Dolma2 tokenizer and OLMo 2 chat template;
3. verifies OLMo can read the raw token and assistant-label-mask shards;
4. runs one `OLMo2-370M` optimization step at the real SFT per-rank shape (`8×4096`) with bf16, FlashAttention-2, fused linear CE, full activation checkpointing, `torch.compile`, and AdamW.

It deliberately uses synthetic conversations and random model weights. Colab has no eduLLM S3 credentials, so this does **not** test the private published dataset, pretraining-checkpoint download, or 8-GPU HSDP/NCCL.

Select **Runtime → Change runtime type → A100 GPU**, then run cells in order.

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime first"
props = torch.cuda.get_device_properties(0)
print(f"device: {props.name}")
print(f"memory: {props.total_memory / 1024**3:.1f} GiB")
print(f"capability: {props.major}.{props.minor}")
print(f"torch: {torch.__version__}  cuda: {torch.version.cuda}")
assert props.major >= 8, "Use an A100 or newer GPU for FlashAttention-2/bfloat16"
assert props.total_memory >= 39 * 1024**3, "Select the 40 GiB A100 runtime for this shape"

## Clone the experiment branch and apply the local fix under test

Re-running this cell hard-resets the Colab clone to the current remote branch. If the gzip-cache fix has not been pushed yet, the cell applies the same minimal change to the Colab checkout and prints that fact. It then asserts that the marker is present, so an older runtime cannot silently test stale code.

In [ ]:
from pathlib import Path

REPO = Path("/content/OLMo-core")
BRANCH = "edullm/frontload-cl"
REMOTE = "https://github.com/edu-llm/OLMo-core.git"

if (REPO / ".git").is_dir():
    %cd {REPO}
    !git remote set-url origin {REMOTE}
    !git fetch --depth 1 origin {BRANCH}
    !git checkout -B {BRANCH} origin/{BRANCH}
    !git reset --hard origin/{BRANCH}
    !git clean -fd
else:
    !git clone --depth 1 --branch {BRANCH} {REMOTE} {REPO}
    %cd {REPO}

print("Branch commit before the notebook-only patch:")
!git rev-parse HEAD
!git log -1 --oneline

source_path = REPO / ".edullm/frontload_cl/sft_tokenize.py"
source = source_path.read_text(encoding="utf-8")
marker = 'is_gzip = raw.read(2) == b"\\x1f\\x8b"'
if marker not in source:
    old = '''        local = _local_path(path)
        opener = gzip.open if local.name.endswith(".gz") else open
'''
    new = '''        local = _local_path(path)
        # cached_path stores remote objects under hash-only names, so inspect the bytes.
        with local.open("rb") as raw:
            is_gzip = raw.read(2) == b"\\x1f\\x8b"
        opener = gzip.open if is_gzip else open
'''
    assert old in source, "The source changed; update this notebook patch instead of guessing"
    source_path.write_text(source.replace(old, new, 1), encoding="utf-8")
    print("Applied the uncommitted gzip magic-byte fix to the Colab checkout")
else:
    print("The branch already contains the gzip magic-byte fix")

assert marker in source_path.read_text(encoding="utf-8")
!grep -n "is_gzip = raw.read" .edullm/frontload_cl/sft_tokenize.py

## Install the platform-matching training stack

The platform image uses PyTorch 2.9 and the prebuilt FlashAttention 2.8.3 wheel. Colab may start with a newer incompatible PyTorch build. If this cell pins PyTorch, it intentionally stops and asks you to **Runtime → Restart session**.

After restarting, **rerun the GPU check, clone cell, and this setup cell**. Do not continue to preprocessing until this cell prints `training dependencies import successfully`; the editable `olmo_core` install happens only on that second pass.

`liger-kernel` is required because the experiment uses fused linear cross-entropy rather than materializing a roughly 12.2 GiB fp32 logits tensor at the SFT rank shape.

In [ ]:
import importlib
import subprocess
import sys

import torch


def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])


print("Colab torch before setup:", torch.__version__, "cuda:", torch.version.cuda)
if not torch.__version__.startswith("2.9."):
    pip(
        "torch==2.9.0+cu128",
        "torchvision==0.24.0+cu128",
        "torchaudio==2.9.0+cu128",
        "--index-url",
        "https://download.pytorch.org/whl/cu128",
    )
    raise SystemExit(
        "PyTorch 2.9 was installed. Restart the session, then rerun the GPU check, "
        "clone cell, and this setup cell. Do not continue until this cell reports success."
    )

%cd /content/OLMo-core
pip("-e", ".[transformers]", "--no-cache-dir")
pip("liger-kernel", "--no-cache-dir")

# pip's editable install adds a .pth file, but the running kernel does not
# reprocess newly created .pth files. Add the src layout explicitly now.
repo_src = "/content/OLMo-core/src"
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
if importlib.util.find_spec("olmo_core") is None:
    raise RuntimeError("olmo_core is still unavailable; inspect the pip output above")
import olmo_core

print("olmo_core:", olmo_core.__file__)

try:
    import flash_attn
    print("flash_attn already installed:", flash_attn.__version__)
except Exception:
    abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheel = (
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
        f"flash_attn-2.8.3+cu12torch2.9cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
    )
    pip(wheel, "--no-build-isolation", "--no-cache-dir")

importlib.import_module("flash_attn")
importlib.import_module("liger_kernel.ops.fused_linear_cross_entropy")
from transformers import AutoTokenizer

print("torch:", torch.__version__)
print("flash_attn:", importlib.import_module("flash_attn").__version__)
print("transformers:", importlib.import_module("transformers").__version__)
print("training dependencies import successfully")

## Reproduce the exact failed preprocessing condition

The failed jobs downloaded `train-00000.jsonl.gz` into a cache file whose local name was only a hash. The old reader looked at that hash-only name, opened the gzip bytes as plain UTF-8, and failed on byte `0x8b`.

This cell creates the same extensionless gzip condition, confirms the fixed reader can stream it, then runs the real pinned Dolma2 tokenizer. It writes enough conversations for at least one `8×4096` batch.

In [ ]:
import gzip
import itertools
import json
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "/content/OLMo-core/.edullm")
from frontload_cl.sft_tokenize import (
    iter_conversation_rows,
    tokenize_conversations_to_dir,
)

WORK = Path("/content/frontload-sft-smoke")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

# Deliberately no .gz suffix: this matches cached_path's hash-only local filename.
CACHED_CONVERSATIONS = WORK / "da630c33b451f2092507d07762de721b7"
with gzip.open(CACHED_CONVERSATIONS, "wt", encoding="utf-8") as fh:
    for i in range(3000):
        row = {
            "source": "colab-synthetic",
            "messages": [
                {"role": "system", "content": "Answer clearly and show the important reasoning."},
                {
                    "role": "user",
                    "content": f"Problem {i}: If a box has {i % 17 + 3} rows of 7 objects, how many objects are there?",
                },
                {
                    "role": "assistant",
                    "content": f"There are {i % 17 + 3} rows with 7 objects each, so multiply: "
                    f"({i % 17 + 3}) × 7 = {(i % 17 + 3) * 7}. The answer is {(i % 17 + 3) * 7} objects.",
                },
            ],
        }
        fh.write(json.dumps(row) + "\n")

assert CACHED_CONVERSATIONS.read_bytes()[:2] == b"\x1f\x8b"
preview = list(itertools.islice(iter_conversation_rows([str(CACHED_CONVERSATIONS)]), 2))
assert len(preview) == 2
print("extensionless gzip streamed successfully; first source:", preview[0]["source"])

stats = tokenize_conversations_to_dir(
    [str(CACHED_CONVERSATIONS)],
    WORK / "tokens",
    max_seq_length=4096,
    tokens_per_shard=1_000_000,
)
print(json.dumps({k: v for k, v in stats.items() if not k.endswith("paths")}, indent=2))
assert stats["num_input_conversations"] == 3000
assert stats["trainable_tokens"] > 0

## Verify the exact OLMo shard reader

OLMo files conventionally end in `.npy`, but this loader expects headerless raw memmaps. This check catches the earlier `np.save()` defect by validating byte lengths and then packing/reading the shards through `NumpyPackedFSLDatasetConfig`, the same dataset class used by `train_sft.py`.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np

if importlib.util.find_spec("olmo_core") is None:
    raise RuntimeError(
        "olmo_core is not installed. After the PyTorch restart, rerun the setup cell and "
        "wait for 'training dependencies import successfully' before continuing."
    )

from olmo_core.data import (
    NumpyDatasetDType,
    NumpyPackedFSLDatasetConfig,
    TokenizerConfig,
)
from olmo_core.data.types import LongDocStrategy

for token_path, mask_path in zip(stats["token_paths"], stats["mask_paths"]):
    token_path = Path(token_path)
    mask_path = Path(mask_path)
    assert not token_path.read_bytes().startswith(b"\x93NUMPY"), "token file has an np.save header"
    token_count = token_path.stat().st_size // np.dtype(np.uint32).itemsize
    mask_count = mask_path.stat().st_size // np.dtype(np.bool_).itemsize
    assert token_count == mask_count, (token_count, mask_count)

packed_config = NumpyPackedFSLDatasetConfig(
    paths=stats["token_paths"],
    label_mask_paths=stats["mask_paths"],
    tokenizer=TokenizerConfig.dolma2(),
    dtype=NumpyDatasetDType.uint32,
    sequence_length=4096,
    work_dir=str(WORK / "packed-cache"),
    generate_doc_lengths=True,
    long_doc_strategy=LongDocStrategy.truncate,
)
dataset = packed_config.build()
dataset.prepare()
print("packed 4096-token instances:", len(dataset))
assert len(dataset) >= 8, "Generate more conversations before the 8-sequence training step"
sample = dataset[0]
assert sample["input_ids"].shape == sample["label_mask"].shape == (4096,)
assert bool(sample["label_mask"].any()), "assistant label mask is empty"
print("first instance trainable tokens:", int(sample["label_mask"].sum()))
print("raw shards round-trip through OLMo successfully")

## One real SFT-shaped optimization step

This uses eight packed 4096-token sequences—the production SFT rank microbatch. It applies assistant-only masks from the generated shards and exercises the 370M model, bf16, FlashAttention-2, fused linear CE, full activation checkpointing, `torch.compile`, backward, gradient clipping, and AdamW.

A single-GPU model holds the full optimizer state, whereas production HSDP shards it over eight GPUs. This Colab step is therefore stricter for parameter/optimizer memory, but it does not test distributed collectives or checkpoint loading. The first compiled step can take several minutes.

In [ ]:
import json
import os
import time

import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"
from frontload_cl.attn import resolve_attn_backend
from olmo_core.config import DType
from olmo_core.data.utils import get_labels
from olmo_core.nn.lm_head import LMLossImplementation
from olmo_core.nn.transformer import TransformerConfig
from olmo_core.nn.transformer.config import TransformerActivationCheckpointingMode

SEQUENCES = 8
SEQ_LENGTH = 4096
COMPILE = True

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

items = [dataset[i] for i in range(SEQUENCES)]
batch = {
    "input_ids": torch.stack([item["input_ids"] for item in items]),
    "label_mask": torch.stack([item["label_mask"] for item in items]),
}
labels = get_labels(batch).cuda(non_blocking=True)
input_ids = batch["input_ids"].cuda(non_blocking=True)
trainable = int((labels != -100).sum())
assert trainable > 0

model_config = TransformerConfig.olmo2_370M(
    vocab_size=TokenizerConfig.dolma2().padded_vocab_size(),
    attn_backend=resolve_attn_backend("flash_2"),
    dtype=DType.bfloat16,
)
model_config.lm_head.loss_implementation = LMLossImplementation.fused_linear
base_model = model_config.build(init_device="cuda")
base_model.apply_activation_checkpointing(TransformerActivationCheckpointingMode.full)
base_model.train()
assert base_model.lm_head is not None
assert base_model.lm_head.loss_implementation == LMLossImplementation.fused_linear

optimizer = torch.optim.AdamW(
    base_model.parameters(),
    lr=8e-5,
    betas=(0.9, 0.95),
    weight_decay=0.0,
)
model = torch.compile(base_model) if COMPILE else base_model

started = time.monotonic()
optimizer.zero_grad(set_to_none=True)
out = model(input_ids=input_ids, labels=labels, return_logits=False)
assert out.logits is None, "fused linear CE unexpectedly returned full logits"
loss = out.loss
assert torch.isfinite(loss), loss
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)
optimizer.step()
torch.cuda.synchronize()

result = {
    "ok": True,
    "loss": float(loss.detach()),
    "grad_norm": float(grad_norm),
    "trainable_tokens": trainable,
    "shape": [SEQUENCES, SEQ_LENGTH],
    "compiled": COMPILE,
    "attention": "flash_2",
    "loss_implementation": str(base_model.lm_head.loss_implementation),
    "seconds": round(time.monotonic() - started, 2),
    "peak_mem_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    "gpu": torch.cuda.get_device_name(0),
}
print(json.dumps(result, indent=2))
assert result["peak_mem_gib"] < 39.5, "The one-GPU smoke exceeded A100 memory"

## Interpreting the result

Success is the final JSON with `"ok": true`, finite loss/gradient norm, `flash_2`, `fused_linear`, and peak memory below the A100 limit.

- A `UnicodeDecodeError` near byte `0x8b` means the clone is stale; rerun the clone cell and confirm its `grep` finds `is_gzip = raw.read(2)`.
- A token/mask size mismatch or `NUMPY` header assertion means the raw-shard writer regressed.
- A FlashAttention import error means PyTorch/FA2 versions do not match; rerun setup after restarting.
- A CUDA OOM here is actionable, but note that full AdamW on one GPU is stricter than eight-way HSDP optimizer memory.

Passing this notebook proves the local gzip-cache, tokenizer, chat formatting, assistant masks, raw OLMo reader, and one-GPU optimization path. It still does not prove S3 access, pretraining-checkpoint loading, or eight-rank HSDP/NCCL. Copy the full output—including the commit hash and final JSON—back into the chat.